# ResNet-18 Clean Baseline

This notebook trains a pretrained ResNet-18 on the clean CIFAKE dataset.

- Label `0`: real
- Label `1`: AI-generated/fake
- Input size: 224 x 224
- Model selection: highest validation F1
- Dataset: already downloaded under `data/raw/`

## 1. Connect to the repository and persistent storage

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/ai-image-detector')
os.chdir(PROJECT_ROOT)

print('Working directory:', Path.cwd())
print('Dataset exists:', Path('data/raw/').exists())
print('Split file exists:', Path('data/splits.csv').exists())
print('Training script exists:', Path('src/train_resnet.py').exists())

## 2. Confirm the data manifest

The training script uses the fixed manifest created during data inspection. It does not create a new random split.

In [ ]:
import pandas as pd

splits = pd.read_csv('data/splits.csv')

display(splits.head())
display(splits.groupby(['split', 'label']).size())

required_columns = {'image_path', 'label', 'split'}
missing_columns = required_columns - set(splits.columns)
assert not missing_columns, f'Missing columns: {missing_columns}'

def resolve_path(path_string):
    path = Path(path_string)
    return path if path.exists() else PROJECT_ROOT / path

missing_images = [
    str(resolve_path(path))
    for path in splits['image_path']
    if not resolve_path(path).exists()
]

print('Missing image files:', len(missing_images))
assert not missing_images, missing_images[:10]

## 3. Check one transformed batch

This confirms that the existing clean transform produces the expected ResNet input shape before a long training run.

In [ ]:
import sys
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, str(PROJECT_ROOT))
from src.transforms import clean_transform

class ManifestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(resolve_path(row['image_path'])).convert('RGB')
        image = self.transform(image)
        return image, int(row['label'])

quick_dataset = ManifestDataset(
    splits[splits['split'] == 'train'],
    clean_transform,
)
quick_loader = DataLoader(quick_dataset, batch_size=4, shuffle=True)
images, labels = next(iter(quick_loader))

print('Batch shape:', images.shape)
print('Labels:', labels.tolist())
assert images.shape == (4, 3, 224, 224)

## 4. Train ResNet-18

The reusable implementation is in `src/train_resnet.py`. The first two epochs train the new final classifier, then the full ResNet-18 is fine-tuned with a smaller learning rate.

In [ ]:
!python src/train_resnet.py \
    --project_root . \
    --splits_file data/splits.csv \
    --output_dir results/resnet18_clean \
    --checkpoint_dir /content/drive/MyDrive/ai-image-detector/checkpoints/resnet18_clean \
    --epochs 10 \
    --freeze_epochs 2 \
    --batch_size 64 \
    --learning_rate 0.0001 \
    --fine_tune_learning_rate 0.00001 \
    --seed 42

## 5. Plot training history

In [ ]:
import matplotlib.pyplot as plt

history = pd.read_csv('results/resnet18_clean/training_history.csv')
display(history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['epoch'], history['train_loss'], marker='o', label='Train')
axes[0].plot(history['epoch'], history['validation_loss'], marker='o', label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['epoch'], history['train_f1'], marker='o', label='Train')
axes[1].plot(history['epoch'], history['validation_f1'], marker='o', label='Validation')
axes[1].set_title('F1 score')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. Display final test metrics and confusion matrix

In [ ]:
import json
import seaborn as sns

with open('results/resnet18_clean/test_metrics.json') as file:
    metrics = json.load(file)

print(json.dumps(metrics, indent=2))

matrix = metrics['confusion_matrix']

plt.figure(figsize=(6, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['real', 'AI/fake'],
    yticklabels=['real', 'AI/fake'],
)
plt.xlabel('Predicted label')
plt.ylabel('Actual label')
plt.title('ResNet-18 Clean Baseline')
plt.show()

## 7. Record the experiment

Record the final accuracy, precision, recall, F1 score, best validation epoch, and confusion matrix in the project report.

The test set was evaluated only after selecting the checkpoint with the best validation F1.